# TODO

- nearest neighbor query
- model selection (1, 5, 10, 15, 20 neighbors)

In [1]:
import numpy as np
import pandas as pd
import pickle

from pathlib import Path
from itertools import product
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from numpy.lib.stride_tricks import sliding_window_view

from aipad.pad_imputation import (
    make_train_test, form_test_matrices, calculate_scores,
    plot_results, read_npzs, save_results, mean_rebinning
)
from aipad.spacecrafts import SoloConstants, WindConstants

solo = SoloConstants()
wind = WindConstants()

In [2]:
bin_width_deg = 1
time_avg_min = 1
avg_bin_dir = f"{time_avg_min}min_{bin_width_deg}deg"

load_path = Path("./data/intensities") / avg_bin_dir
cov_path = Path("./data/coverages") / avg_bin_dir
plot_path = Path(f"./plots/")
model_path = Path("./data/models")

plot_path.mkdir(exist_ok=True)
model_path.mkdir(exist_ok=True)

In [3]:
# Train test split: pick e.g. every 3rd as test (train, train, test, train, train, test...)
# Events are in time order so this takes the solar cycle into account.
# NOTE: using future observations to predict past observations is generally not OK, but
# the nature of the data is such that this can be ignored.

hist_arr, reduced_hist_arr, intensity_arr, metadata_arr = read_npzs(load_path=load_path)

X_train, X_test, y_train, y_test, I_train, I_test, meta_train, meta_test = make_train_test(hist_arr, reduced_hist_arr, intensity_arr, metadata_arr, n=3)

In [ ]:

# Stacking of training cases for KNN imputer:
# concatenate 5 successive rows to one feature vector (5 * 180 = 900 features)
trains = []
for train in X_train:
    train_stacked = sliding_window_view(train, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    trains.append(train_stacked)

X_train_stacked = np.vstack(trains)

### Mean value imputation
Two ways to do this with either rows as samples or columns as samples. `SimpleImputer()` uses mean of column as the value to fill.

TODO/Ideas:
- Dividing into smaller chunks probably makes it better (in both cases), since then the variations over large scales don't affect the local mean

In [5]:
# fitting to test cases since "fitting" is just calculating the means of each column. "Supervised learning" approach is not applicable here
# Columns as samples: each time bin is filled with the mean across the whole angle space.
for j in range(0, len(y_test)):
    model = SimpleImputer(keep_empty_features=True) 
    res = form_test_matrices(model, X_test[j], y_test[j], transpose=True)
    plot_results(model, res, I_test[j], sc=wind, cov_sc=solo, save_path=Path("./plots") / "meanimputer" / f"results_{j}_columns-as-samples.png")

TypeError: plot_results() missing 1 required positional argument: 'intensities'

### KNNImputer

In [ ]:
neighbors = 5

model = KNNImputer(n_neighbors=neighbors, weights="distance", keep_empty_features=True)
model.fit(X_train_stacked)

In [ ]:
for j in range(0, len(y_test)):
    res = form_test_matrices(model, X_test[j], y_test[j])
    plot_results(model, res, "mse", I_test[j], sc=wind, cov_sc=solo, save_path=plot_path / f"results_{j}.png")

In [6]:
def form_test_matrices(model: KNNImputer, X_test: np.ndarray,
                       y_test: np.ndarray, n_samples=10, sample_loc=120) -> list:
    true = X_test[sample_loc:sample_loc+n_samples]
    test = y_test[sample_loc:sample_loc+n_samples]
    reduced = np.where(np.isfinite(test), true, np.nan)
    
    reduced_reshaped = reduced.reshape((2, 900))
    pred_full_reshaped = model.transform(reduced_reshaped)
    pred_full = pred_full_reshaped.reshape((10, 180))

    pred = np.where((np.isnan(reduced)
                    & np.isfinite(true)
                    & np.meshgrid(np.isfinite(reduced).any(axis=1), np.arange(0, reduced.shape[1]),
                                  indexing="ij")[0]),
                    pred_full, np.nan)
    target = np.where(np.isfinite(pred), true, np.nan)

    return [true, reduced, pred_full, pred, target]

In [ ]:
import logging
import sys
filehandler = logging.FileHandler(filename="./logs/cv_results.log", encoding="utf-8")
streamhandler = logging.StreamHandler(sys.stdout)
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', datefmt='%m/%d/%Y %H:%M:%S', handlers=[filehandler, streamhandler], force=True)


# Model selection CV
neighbors = [1, 5, 10, 15]
weights = ["distance", "uniform"]
n_splits = 3
n_events = 20
n_samples = 10
sample_loc = 120 # onset is located at two hours after start of the window

# Cross validation: shift train-test split by one 3 times (a sort of 3-fold CV),
# but test only 10 samples in each event (after onset) and only 20 events in each fold

for n, w in product(neighbors, weights):
    model = KNNImputer(n_neighbors=n, weights=w, keep_empty_features=True)
    
    for shift in range(n_splits):
        logging.info(f"KNN neighbors={n}, weights={w}, split {shift}")
        result_df = pd.DataFrame()
        (X_train, X_test, y_train, y_test,
        I_train, I_test, meta_train, meta_test) = make_train_test(hist_arr, reduced_hist_arr,
                                                                intensity_arr, metadata_arr,
                                                                n=n_splits, shift=shift)
        trains = []
        for train in X_train:
            train_stacked = sliding_window_view(train, window_shape=(5, train.shape[1])).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
            trains.append(train_stacked)

        X_train_stacked = np.vstack(trains)
        model.fit(X_train_stacked)
        scores = []
        for j in range(20):
            res = form_test_matrices(model, X_test[j], y_test[j])
            # plot_results(model, res, "rmse", I_test[j], sc=wind, cov_sc=solo,
            #              save_path=plot_path / "knnimputer" / "crossvalidation" / f"knn_{n}_{w}_split_{shift}_results_{j}.png")
            
            # Score on n_samples samples during main event
            score = calculate_scores(res[3],
                                     res[4], "rmse")
            result_df = save_results(model, res, meta_test[j], "rmse", Path(f"./knn_{n}_{w}_split_{shift}_results.csv"))
            scores.append(score)
            logging.info(f"Score for test event {j}: {score}")

        clear_output()
        logging.info(f"KNN neighbors={n}, weights={w}: CV score on fold {shift}={np.mean(scores):.4f}")


02/10/2026 10:50:04 KNN neighbors=1, weights=distance, split 0


In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(max_features=32)

# X = true, y = reduced. Train model with y as input, X as output
X_trains = []
y_trains = []
for X, y in zip(X_train, y_train):
    X_stacked = sliding_window_view(X, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])
    y_stacked = sliding_window_view(y, window_shape=(5,180)).reshape(-1, 5 * train.shape[1])   # ChatGPT solution
    y_trains.append(y_stacked)
    X_trains.append(X_stacked)

X_train_stacked = np.vstack(X_trains)
y_train_stacked = np.vstack(y_trains)

In [1]:
from sklearn.metrics import nan_euclidean_distances
from sklearn.impute import KNNImputer